# Extracting OISST timeseries from green crab study locations.

First import libraries.

In [1]:
import re
from getpass import getpass

import icechunk as ic
import numpy as np
import xarray as xr
import pandas as pd
from pydantic import BaseModel, Field
import geopandas as gpd

Open and explore the source dataframe (once it was externally pruned down to just sites/location).

In [2]:
source_df = pd.read_csv("./Green_Crab_MT_points.csv", engine="python")
source_df

,SITES,LATITUDE,LONGITUDE
0,Little John-NW,43.753894,-70.136267
1,Little John-NE,43.753994,-70.135828
2,Little John-SE,43.753647,-70.136350
3,Mackworth- SE,43.6919,-70.228983
4,Mackworth- SW,43.691739,-70.228897
...,...,...,...
1263,Long Point Cove,43.7818491,-69.934574
1264,Lowell Cove,43.7509752,-69.983197
1265,Skitterygusset,43.71458,-70.247910
1266,Mackworth Island - Beach,43.68958,-70.235720


Filter to just unique points (1,268 -> 85 points), and set the site names as the index variable (so they become the coordinate variable).

In [3]:
unique_df = source_df.drop_duplicates(["SITES"]).set_index("SITES")
unique_df

,LATITUDE,LONGITUDE
SITES,,
Little John-NW,43.753894,-70.136267
Little John-NE,43.753994,-70.135828
Little John-SE,43.753647,-70.136350
Mackworth- SE,43.6919,-70.228983
Mackworth- SW,43.691739,-70.228897
...,...,...
Lowell Cove,43.7509752,-69.983197
Garrison Cove,43.750853,-69.959060
Orrs Cove,43.835471,-69.913802


Convert to an Xarray dataset with the point names as a coordinate, and rename the columns to what other operations expect (points, lat, lon).

In [4]:
points_ds = unique_df.to_xarray().rename({"LATITUDE": "lat", "LONGITUDE": "lon", "SITES": "points"})
points_ds

<xarray.Dataset> Size: 2kB
Dimensions:  (points: 85)
Coordinates:
  * points   (points) object 680B 'Little John-NW' ... 'Stovers Point'
Data variables:
    lat      (points) object 680B '43.753894' '43.753994' ... '43.7578601'
    lon      (points) float64 680B -70.14 -70.14 -70.14 ... -69.91 -69.99 -70.0

Enter in AWS access credentials for the dataset. The daily dataset itself is in a public bucket, but the metadata and the monthly data is in a NERACOOS requester pays bucket.

In [5]:
access_key = getpass("Enter S3 access key ID: ")
secret_key = getpass("Enter S3 secret access key: ")

Enter S3 access key ID:  ········
Enter S3 secret access key:  ········


Some scaffolding to help opening the dataset. The daily data lives in `noaa-cdr-sea-surface-temp-optimum-interpolation-pds`, while the metadata and monthly data is in `neracoos-data-requester-pays`, which makes the access a little convoluted.

In [6]:
BUCKET = "noaa-cdr-sea-surface-temp-optimum-interpolation-pds"
URL_PREFIX = f"s3://{BUCKET}/"
DATA_PREFIX = "data/v2.1/avhrr"
REGION = "us-east-1"

# obstore / icechunk want the prefix without a trailing slash for the registry
# key, while icechunk's VirtualChunkContainer matches against the trailing-slash
# form (mirroring services/xreds dataset_spec.py).
STORE_PREFIX = URL_PREFIX.rstrip("/")


def open_repo(prefix: str) -> ic.Repository:
    """Open (or create on first run) the icechunk repo for a store prefix, with
    the NOAA virtual chunk container registered and authorized."""
    config = ic.RepositoryConfig.default()
    config.set_virtual_chunk_container(
        ic.VirtualChunkContainer(
            url_prefix=URL_PREFIX,
            store=ic.s3_store(region=REGION, anonymous=True),
        ),
    )

    return ic.Repository.open(
        ic.s3_storage(
            bucket="neracoos-data-requester-pays",
            prefix=prefix,
            region="us-east-1",
            access_key_id=access_key,
            secret_access_key=secret_key,
            requester_pays=True,
        ),
        config=config,
        authorize_virtual_chunk_access=ic.containers_credentials(
            {URL_PREFIX: ic.s3_anonymous_credentials()}
        ),
    )

Now we can access the metadata, and open a session for access. There are a few steps here because [Icechunk](https://icechunk.io/en/stable/) uses a strong versioning system like Git, but we can use the latest data on `main`.

In [7]:
final_repo = open_repo("oisst/final")
final_repo

<icechunk.Repository (v2)>
storage:
    <icechunk.Storage>
    type: S3 (native)
    bucket: neracoos-data-requester-pays
    prefix: oisst/final
    region: us-east-1
    requester_pays: True
config: <RepositoryConfig ...>

In [8]:
final_session = final_repo.readonly_session("main")
final_session

<icechunk.session.Session>
read_only: True
snapshot_id: ZFAQBYGWY6YVZCY4TMC0

Now we open the OISST datatree. The daily data has the 4 normal variables, but the monthly data has those variables aggregated ito mean, min, max, and standard deviation. We're going to use the monthly dataset.

In [9]:
oisst_dt = xr.open_datatree(final_session.store, engine="zarr", zarr_version=3)
oisst_dt

/home/.pixi/envs/default/lib/python3.14/site-packages/argopy/utils/lists.py:38: UserWarning: An error occurred while loading the ERDDAP data fetcher, it will not be available !
<class 'ImportError'>
cannot import name '_quote_string_constraints' from 'erddapy.erddapy' (/home/.pixi/envs/default/lib/python3.14/site-packages/erddapy/erddapy.py)
  warnings.warn(
/home/.pixi/envs/default/lib/python3.14/site-packages/argopy/utils/lists.py:50: UserWarning: An error occurred while loading the ArgoVis data fetcher, it will not be available !
<class 'ImportError'>
cannot import name '_quote_string_constraints' from 'erddapy.erddapy' (/home/.pixi/envs/default/lib/python3.14/site-packages/erddapy/erddapy.py)
  warnings.warn(
/home/.pixi/envs/default/lib/python3.14/site-packages/argopy/utils/lists.py:66: UserWarning: An error occurred while loading the GDAC data fetcher, it will not be available !
<class 'ImportError'>
cannot import name '_quote_string_constraints' from 'erddapy.erddapy' (/home/.pi

<xarray.DataTree>
Group: /
├── Group: /daily
│       Dimensions:  (time: 16405, zlev: 1, lat: 720, lon: 1440)
│       Coordinates:
│         * lat      (lat) float32 3kB -89.88 -89.62 -89.38 -89.12 ... 89.38 89.62 89.88
│         * zlev     (zlev) float32 4B 0.0
│         * time     (time) datetime64[ns] 131kB 1981-12-21T12:00:00 ... 2026-08-02T1...
│         * lon      (lon) float32 6kB 0.125 0.375 0.625 0.875 ... 359.4 359.6 359.9
│       Data variables:
│           err      (time, zlev, lat, lon) float64 136GB ...
│           anom     (time, zlev, lat, lon) float64 136GB ...
│           sst      (time, zlev, lat, lon) float64 136GB ...
│           ice      (time, zlev, lat, lon) float64 136GB ...
│       Attributes: (12/38)
│           title:                      NOAA/NCEI 1/4 Degree Daily Optimum Interpolat...
│           source:                     ICOADS, NCEP_GTS, GSFC_ICE, NCEP_ICE, Pathfin...
│           id:                         oisst-avhrr-v02r01.20260802.nc
│           naming_authority:           gov.noaa.ncei
│           summary:                    NOAAs 1/4-degree Daily Optimum Interpolation ...
│           cdm_data_type:              Grid
│           ...                         ...
│           ncei_template_version:      NCEI_NetCDF_Grid_Template_v2.0
│           comment:                    Data was converted from NetCDF-3 to NetCDF-4 ...
│           sensor:                     Thermometer, AVHRR
│           Conventions:                CF-1.6, ACDD-1.3
│           references:                 Reynolds, et al.(2007) Daily High-Resolution-...
│           Description:                Reynolds, et al.(2007) Daily High-resolution ...
└── Group: /monthly
        Dimensions:       (time: 537, zlev: 1, lat: 720, lon: 1440)
        Coordinates:
          * lat           (lat) float32 3kB -89.88 -89.62 -89.38 ... 89.38 89.62 89.88
          * time          (time) datetime64[ns] 4kB 1981-10-01 1985-03-01 ... 2026-05-01
          * zlev          (zlev) float32 4B 0.0
          * lon           (lon) float32 6kB 0.125 0.375 0.625 ... 359.4 359.6 359.9
        Data variables: (12/17)
            anom_max      (time, zlev, lat, lon) float64 4GB ...
            anom_mean     (time, zlev, lat, lon) float64 4GB ...
            anom_min      (time, zlev, lat, lon) float64 4GB ...
            err_mean      (time, zlev, lat, lon) float64 4GB ...
            err_std       (time, zlev, lat, lon) float64 4GB ...
            ice_max       (time, zlev, lat, lon) float64 4GB ...
            ...            ...
            sst_min       (time, zlev, lat, lon) float64 4GB ...
            ice_std       (time, zlev, lat, lon) float64 4GB ...
            err_min       (time, zlev, lat, lon) float64 4GB ...
            sst_std       (time, zlev, lat, lon) float64 4GB ...
            days_missing  int64 8B ...
            ice_mean      (time, zlev, lat, lon) float64 4GB ...

We're explictly sorting the time as our aggregation process for both daily and monthly data is non linear, and we'd rather not have miss-ordered dates later.

In [10]:
print(f"Full dataset: {oisst_dt.nbytes / 1_000_000_000:,.2f} Gigabytes")

Full dataset: 615.54 Gigabytes


In [11]:
monthly_ds = oisst_dt["monthly"].to_dataset()
# Or directly with
# monthly_ds = xr.open_zarr(final_session.store, group="monthly", zarr_version=3)
monthly_ds = monthly_ds.sortby("time")
monthly_ds

<xarray.Dataset> Size: 71GB
Dimensions:       (time: 537, zlev: 1, lat: 720, lon: 1440)
Coordinates:
  * lat           (lat) float32 3kB -89.88 -89.62 -89.38 ... 89.38 89.62 89.88
  * time          (time) datetime64[ns] 4kB 1981-10-01 1981-11-01 ... 2026-06-01
  * zlev          (zlev) float32 4B 0.0
  * lon           (lon) float32 6kB 0.125 0.375 0.625 ... 359.4 359.6 359.9
Data variables: (12/17)
    anom_max      (time, zlev, lat, lon) float64 4GB ...
    anom_mean     (time, zlev, lat, lon) float64 4GB ...
    anom_min      (time, zlev, lat, lon) float64 4GB ...
    err_mean      (time, zlev, lat, lon) float64 4GB ...
    err_std       (time, zlev, lat, lon) float64 4GB ...
    ice_max       (time, zlev, lat, lon) float64 4GB ...
    ...            ...
    sst_min       (time, zlev, lat, lon) float64 4GB ...
    ice_std       (time, zlev, lat, lon) float64 4GB ...
    err_min       (time, zlev, lat, lon) float64 4GB ...
    sst_std       (time, zlev, lat, lon) float64 4GB ...
    days_missing  int64 8B ...
    ice_mean      (time, zlev, lat, lon) float64 4GB ...

In [12]:
print(f"Monthly: {monthly_ds.nbytes / 1_000_000_000:,.2f} bytes")

Monthly: 71.27 bytes


Most of these samples were collected on shore, so we need to find the nearest 'wet' OISST cell. This also handles when our point longitudes are in -180 to 180 vs OISST in 0-360.

In [13]:
def sel_ocean_points(da, pts, window=1.0):
    """Pointwise-select `da` at each (lat, lon) in `pts`, snapping to ocean cells.

    Handles two gotchas of point extraction from OISST-style grids:

    - **Longitude convention** - converts the points' -180..180 longitudes to
      0..360 when that's what the grid uses (detected from `da.lon.max()`).
    - **Land cells** - a point whose nearest cell is all-null (land) is snapped
      to the nearest non-null (ocean) cell within +/-`window` degrees, using
      the first timestep's null mask as the land/ocean mask.

    Returns a DataArray with a `points` dimension (labels from `pts`), the
    chosen cell's lat/lon as coordinates, and a boolean `snapped` coordinate
    marking points that were moved off land.
    """
    lon_max = float(da.lon.max())
    lons = pts["lon"] % 360 if lon_max > 180 else pts["lon"]
    ocean = da.isel(time=0).notnull().drop_vars("time", errors="ignore")

    series = []
    snapped = []
    for i in range(pts.sizes["points"]):
        t_lat, t_lon = float(pts["lat"][i]), float(lons[i])
        cell = da.sel(lat=t_lat, lon=t_lon, method="nearest")
        if bool(cell.isnull().all()):
            win = da.sel(
                lat=slice(t_lat - window, t_lat + window),
                lon=slice(t_lon - window, t_lon + window),
            )
            dist = (win.lat - t_lat) ** 2 + (win.lon - t_lon) ** 2
            idx = dist.where(ocean.sel(lat=win.lat, lon=win.lon)).argmin(
                dim=("lat", "lon")
            )
            cell = win.isel(idx)
            snapped.append(True)
        else:
            snapped.append(False)
        series.append(cell)

    out = xr.concat(series, dim=pts["points"])
    return out.assign_coords(snapped=("points", snapped))

Now that function helps us [pointwise index](https://docs.xarray.dev/en/latest/user-guide/indexing.html#more-advanced-indexing) into the OISST data. Now points and time become the dimensions of the dataset (instead of time/lat/lon), and there is a point for each one that we entered.

_Note: This cell is disabled as it takes about 40 minutes to run with a good connection, and about 15 GB of data. Click the 3 dots on the cell and re-enable it in the menu to run it and the following cells._

In [14]:
points_sst = sel_ocean_points(
    monthly_ds["sst_mean"].squeeze(drop=True), points_ds
)
points_sst

<xarray.DataArray 'sst_mean' (points: 85, time: 537)> Size: 365kB
array([[11.55580619,  8.94466647,  7.06419339, ...,  5.46333321,
         9.10967722, 13.75933303],
       [11.55580619,  8.94466647,  7.06419339, ...,  5.46333321,
         9.10967722, 13.75933303],
       [11.55580619,  8.94466647,  7.06419339, ...,  5.46333321,
         9.10967722, 13.75933303],
       ...,
       [11.52354813,  8.9799998 ,  7.190645  , ...,  5.31199988,
         8.86225787, 13.3989997 ],
       [11.52354813,  8.9799998 ,  7.190645  , ...,  5.31199988,
         8.86225787, 13.3989997 ],
       [11.52354813,  8.9799998 ,  7.190645  , ...,  5.31199988,
         8.86225787, 13.3989997 ]], shape=(85, 537))
Coordinates:
    lat      (points) float32 340B 43.62 43.62 43.62 43.62 ... 43.62 43.62 43.62
  * time     (time) datetime64[ns] 4kB 1981-10-01 1981-11-01 ... 2026-06-01
    lon      (points) float32 340B 289.9 289.9 289.9 289.9 ... 290.1 290.1 290.1
  * points   (points) object 680B 'Little John-NW' ... 'Stovers Point'
    snapped  (points) bool 85B True True True False ... True True False True
Attributes:
    long_name:  Daily sea surface temperature
    units:      Celsius
    valid_min:  -300
    valid_max:  4500

We save it to NetCDF which keeps the dimensions and metadata, and CSV which can be easier to access.

In [18]:
points_sst.to_netcdf("oisst_points_sst_mean.nc")

In [19]:
points_sst.to_dataframe().to_csv("oisst_points_sst_mean.csv")

points_sst_nc = xr.open_dataset("~/Downloads/oisst_points_sst_mean.nc")
points_sst_nc